# 28. Do the five LightGBM target-encoded seeds still earn a place?

**One variable against ledger row 44** (`stack_logit_29_oof`, CV 0.967925): the member
set goes from twenty-nine to twenty-four by removing `te42`, `te2024`, `te7`, `te2025`
and `te13`. Same combiner, same `C`, same fold-wise protocol, same folds.

This is the first experiment in this competition that makes the stack **smaller**.

## Why

Row 44's coefficients showed those five collapsing once XGBoost and its seeds arrived:

| | row 34 | row 44 |
|---|---|---|
| `te42` | +0.0767 | **-0.0032** |
| `te7` | +0.0612 | **-0.0089** |
| `te13` | +0.0842 | +0.0081 |
| `te2024` | +0.0825 | +0.0033 |
| `te2025` | +0.0819 | +0.0032 |

All five together now carry about +0.0025, against roughly +0.081 each in row 34. Two
are negative. XGBoost occupies their position on their feature set and does it better.

## The methodological hazard, stated before the result

**The removal set was chosen by looking at coefficients fitted on this same
out-of-fold matrix, and the pruned stack is then scored on that same matrix.** That is
selection on the validation set, which is exactly the error this file warned about when
it refused to drop seed 2024 from the raw-feature blend: "Dropping it would be
selecting on validation performance, which is how a CV gets overfit."

Two things make it weaker here than there, and neither makes it disappear.

- It is **one pre-declared hypothesis with a mechanism**, that XGBoost displaced these
  five specifically, rather than a search over subsets. Row 39 predicted this before
  row 44 measured it.
- The five are removed **as a group**, named in advance, not chosen individually by
  score.

**So the claim this notebook can support is "no loss", not "gain".** If the pruned
stack scores materially higher, that number is optimistically biased and must be
reported as suspect rather than banked. If it scores the same, the honest conclusion is
that five members are carrying nothing and the simpler stack is preferable at equal
score, which is a robustness argument for the private leaderboard rather than a CV one.

## The honesty check

The cell after the comparison asks whether the drop decision would have been reached
**from training-fold information alone**. For each fold it refits the 29-member
combiner on the other four folds and reports what those five coefficients were there.
If they are already at zero or negative in all five training-fold fits, the selection
did not need the held-out fold and the bias above is small. If they are healthy in some
folds, the removal is being driven by the very rows it is scored on and the result
should be discarded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]


In [2]:
# Row 32's twenty-three in row 32's order, then the target-encoded neural model.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
    ("cat42", O / "catboost_te_oof.npy", O / "catboost_te_test.npy"),
    ("cat2024", O / "catboost_te_seed2024_oof.npy",
     O / "catboost_te_seed2024_test.npy"),
    ("cat7", O / "catboost_te_seed7_oof.npy", O / "catboost_te_seed7_test.npy"),
    ("cat2025", O / "catboost_te_seed2025_oof.npy",
     O / "catboost_te_seed2025_test.npy"),
    ("cat13", O / "catboost_te_seed13_oof.npy", O / "catboost_te_seed13_test.npy"),
    ("neural_te", O / "neural_te_oof.npy", O / "neural_te_test.npy"),
    ("xgb_te", O / "xgb_te_oof.npy", O / "xgb_te_test.npy"),
    ("xgb2024", O / "xgb_te_seed2024_oof.npy", O / "xgb_te_seed2024_test.npy"),
    ("xgb7", O / "xgb_te_seed7_oof.npy", O / "xgb_te_seed7_test.npy"),
    ("xgb2025", O / "xgb_te_seed2025_oof.npy", O / "xgb_te_seed2025_test.npy"),
    ("xgb13", O / "xgb_te_seed13_oof.npy", O / "xgb_te_seed13_test.npy"),
]
DROP = ["te42", "te2024", "te7", "te2025", "te13"]
NEW = []
CATS = ["cat42", "cat2024", "cat7", "cat2025", "cat13"]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
KEEP24 = [i for i, n in enumerate(names) if n not in DROP]
print(f"{len(names)} members, oof {Loof.shape}, test {Ltest.shape}")
print("member CV:")
for n in names:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    print(f"  {n:10} {cv:.6f}" + ("   <- new" if n in NEW else ""))

29 members, oof (691369, 29), test (296302, 29)
member CV:


  te42       0.966782


  te2024     0.966771


  te7        0.966729


  te2025     0.966743


...

  xgb_te     0.967099


  xgb2024    0.967148


  xgb7       0.967132


  xgb2025    0.967099


  xgb13      0.967111


In [3]:
def run(cols):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)],
                                                           y[tr])
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf


ALL = list(range(len(names)))
per29, test29, coef29 = run(ALL)
per24, test24, coef24 = run(KEEP24)

ROW44_CV = 0.967925
repro = per29.mean() - ROW44_CV
REPRODUCED = abs(repro) < 1e-4

print(f"{'':26} {'fold 0':>9} {'fold 1':>9} {'fold 2':>9} {'fold 3':>9} {'fold 4':>9}")
for lbl, p in (("29 members, row 44", per29), ("24 members, pruned", per24)):
    print(f"{lbl:26} " + " ".join(f"{v:9.6f}" for v in p))
print()
print(f"29-member CV {per29.mean():.6f} +/- {per29.std():.6f}"
      f"   (row 44 recorded {ROW44_CV:.6f}, diff {repro:+.2e})")
print(f"24-member CV {per24.mean():.6f} +/- {per24.std():.6f}   "
      f"[{len(KEEP24)} members, {len(DROP)} removed]")
if not REPRODUCED:
    print("\nROW 44 DID NOT REPRODUCE. Nothing below is comparable to it.")

                              fold 0    fold 1    fold 2    fold 3    fold 4
29 members, row 44          0.967287  0.968045  0.968219  0.968474  0.967601
24 members, pruned          0.967288  0.968051  0.968222  0.968480  0.967605

29-member CV 0.967925 +/- 0.000428   (row 44 recorded 0.967925, diff +1.23e-07)
24-member CV 0.967929 +/- 0.000429   [24 members, 5 removed]


In [4]:
def paired(a, b, lbl):
    d = a - b
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"{lbl:38} {d.mean():+.6f}  sd {d.std(ddof=1):.6f}  "
          f"{(d > 0).sum()}/5  t(4)={t:.2f}")
    print("     per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    return d


d_prune = paired(per24, per29, "24 pruned vs 29 members (row 44)")
print()
print("Positive means the smaller stack scored higher, which per the header is the")
print("case to distrust rather than the case to bank.")

24 pruned vs 29 members (row 44)       +0.000004  sd 0.000002  5/5  t(4)=5.11
     per fold: +0.000002  +0.000006  +0.000003  +0.000006  +0.000004

Positive means the smaller stack scored higher, which per the header is the
case to distrust rather than the case to bank.


In [5]:
# The honesty check. Would the drop decision have been reached without the held-out
# fold? coef29 rows are the per-fold fits, and each was trained on the OTHER four
# folds, so these are training-fold coefficients for the fold in question.
idx = {n: names.index(n) for n in DROP}
print("coefficient of each dropped member, in each fold's training-fold fit:")
print(f"{'member':10} " + " ".join(f"{'fold ' + str(f):>10}" for f in range(5))
      + f" {'max':>10}")
worst = -1e9
for n in DROP:
    row = coef29[:, idx[n]]
    worst = max(worst, row.max())
    print(f"{n:10} " + " ".join(f"{v:>10.4f}" for v in row) + f" {row.max():>10.4f}")

print()
print(f"largest coefficient any dropped member reached in any training-fold fit: "
      f"{worst:+.4f}")
kept = [n for n in names if n not in DROP]
smallest_kept = min(coef29[:, names.index(n)].mean() for n in kept
                    if coef29[:, names.index(n)].mean() > 0)
print(f"smallest positive mean coefficient among the members kept: {smallest_kept:+.4f}")
SELECTION_SAFE = worst < smallest_kept
print()
print("The drop is supported by training-fold information alone"
      if SELECTION_SAFE else
      "WARNING: at least one dropped member looks healthy in a training-fold fit, so")
if not SELECTION_SAFE:
    print("this removal is partly driven by the rows it is scored on. Treat as suspect.")

coefficient of each dropped member, in each fold's training-fold fit:
member         fold 0     fold 1     fold 2     fold 3     fold 4        max
te42           0.0145    -0.0082    -0.0090    -0.0083    -0.0051     0.0145
te2024         0.0015     0.0122    -0.0160     0.0013     0.0173     0.0173
te7           -0.0085    -0.0203    -0.0040     0.0072    -0.0188     0.0072
te2025        -0.0022    -0.0121     0.0013     0.0131     0.0157     0.0157
te13           0.0090     0.0193     0.0041     0.0113    -0.0031     0.0193

largest coefficient any dropped member reached in any training-fold fit: +0.0193
smallest positive mean coefficient among the members kept: +0.0051

this removal is partly driven by the rows it is scored on. Treat as suspect.


In [6]:
mean_p = float(d_prune.mean())
sd_p = float(d_prune.std(ddof=1))
# Equivalence rather than superiority: the claim is that removing five members costs
# nothing. The band is row 44's own accepted gain, +0.000052, since a change smaller
# than the smallest thing this repo has ever accepted is not a change it can act on.
BAND = 5.2e-05

print(f"pruned minus full: {mean_p:+.6f}, paired sd {sd_p:.6f}, "
      f"{(d_prune > 0).sum()}/5 folds")
print(f"equivalence band : +/- {BAND:.6f}  (row 44's accepted gain)")
print()
if not REPRODUCED:
    print("VERDICT: blocked, row 44 did not reproduce")
elif not SELECTION_SAFE:
    print("VERDICT: discard. The removal is selected on the rows it is scored on.")
elif abs(mean_p) < BAND:
    print("VERDICT: EQUIVALENT. Five of twenty-nine members can be removed without a")
    print("  measurable cost. The claim is simplicity at equal score, not a gain, and")
    print("  the header committed to reading it that way before the number existed.")
elif mean_p >= BAND:
    print("VERDICT: the pruned stack scores higher, and per the header this is the")
    print("  suspect direction. Report as biased by selection, do not bank it.")
else:
    print("VERDICT: removing them costs more than row 44's accepted gain. They earn")
    print("  their place after all, and the coefficient collapse was misleading.")

pruned minus full: +0.000004, paired sd 0.000002, 5/5 folds
equivalence band : +/- 0.000052  (row 44's accepted gain)

VERDICT: discard. The removal is selected on the rows it is scored on.


In [7]:
pred = test24.mean(axis=0)
prob = 1 / (1 + np.exp(-pred))
prev = pd.read_csv(S / "stack_oof_29.csv")
assert (prev["id"].to_numpy() == test["id"].to_numpy()).all()
print(f"spearman vs row 44 on disk : "
      f"{pd.Series(pred).corr(pd.Series(prev['addicted_label'].to_numpy()), method='spearman'):.7f}")

out = S / "stack_oof_24pruned.csv"
sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
sub.to_csv(out, index=False)
print(f"\nwrote {out.name}, {len(sub):,} rows")
print(f"ledger: CV {per24.mean():.6f} +/- {per24.std():.6f}, "
      f"vs row 44 {mean_p:+.6f} ({(d_prune > 0).sum()}/5, sd {sd_p:.6f})")

spearman vs row 44 on disk : 0.9999992



wrote stack_oof_24pruned.csv, 296,302 rows
ledger: CV 0.967929 +/- 0.000429, vs row 44 +0.000004 (5/5, sd 0.000002)
